<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Discretización por Kmeans</h1>
        <h3 style="margin: 0 0 8px 0; color: white; font-size: 1.15em; line-height: 1.3;">Ingeniería de Características</h3>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Maestría en Ciencia de Datos</h3>
    </div>
</div>

## Introducción


K-Means es un algoritmo no supervisado de Clustering. Se utiliza cuando tenemos un montón de datos sin etiquetar. El objetivo de este algoritmo es el de encontrar “K” grupos (clusters) entre los datos crudos. 

**¿Cómo funciona?**

El algoritmo trabaja iterativamente para asignar a cada “muestra” uno de los “K” grupos basado en sus características. Son agrupados en base a la similitud de sus features (las columnas). Como resultado de ejecutar el algoritmo tendremos:


> Los `“centroids”` de cada grupo que serán unas “coordenadas” de cada uno de los K conjuntos qu>e se utilizarán para poder etiquetar nuevas muestras.

> `Etiquetas` para el conjunto de datos de entrenamiento. Cada etiqueta perteneciente a uno de los K grupos formados.

Los grupos se van definiendo de manera “orgánica”, es decir que se va ajustando su posición en cada iteración del proceso, hasta que converge el algoritmo. Una vez hallados los centroids deberemos analizarlos para ver cuales son sus características únicas, frente a la de los otros grupos. Estos grupos son las etiquetas que genera el algoritmo.

## Algoritmo K-means

El algoritmo utiliza una proceso **iterativo** en el que se van ajustando los grupos para producir el resultado final. Para ejecutar el algoritmo deberemos pasar como entrada el `conjunto de datos` y un valor de `K`. El conjunto de datos serán las características o features para cada punto. Las posiciones iniciales de los K centroids serán asignadas de manera aleatoria de cualquier punto del conjunto de datos de entrada. Luego se itera en dos pasos:

> 1.- **Paso de asignación** $argmin_{c_i \in C} dist(c_i, x)^2$

> 2.- **Paso de actualización del Centroide**  En este paso los centroides de cada grupo son recalculados. Esto se hace tomando una media de todos los puntos asignados en el paso anterior. $c_i = \frac{1}{|s_i|}\sum_{x_i \in s_i} x_i$

El algoritmo itera entre estos pasos hasta cumplir un criterio de detención:
*  si no hay cambios en los puntos asignados a los grupos,
* o si la suma de las distancias se minimiza,
* o se alcanza un número máximo de iteraciones.

El algoritmo converge a un resultado que puede ser el óptimo local, por lo que será conveniente volver a ejecutar más de una vez con puntos iniciales aleatorios para confirmar si hay una salida mejor.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import make_blobs

In [ ]:
def euclidean_distance(x1, x2):
    return np.sqrt(np.sum((x1-x2)**2))

In [ ]:
class KMeans:

    def __init__(self, K=5, max_iters=100, plot_steps=False):
        self.K = K
        self.max_iters = max_iters
        self.plot_steps = plot_steps

        # lista de indices por cada cluster
        self.clusters = [[] for _ in range(self.K)]

        # centroides de cada cluster 
        self.centroids = []


    def predict(self, X):
        self.X = X
        self.n_samples, self.n_features = X.shape

        # Inicialización
        random_sample_idxs = np.random.choice(self.n_samples, self.K, replace=False)
        self.centroids = [self.X[idx] for idx in random_sample_idxs]

        # Optimizando los clusters
        for _ in range(self.max_iters):
            # asignación de cada punto al centroide más cercano
            self.clusters = self._create_clusters(self.centroids)

            if self.plot_steps:
                self.plot()

            # calculo de los nuevos centroides desde los clusters
            centroids_old = self.centroids
            self.centroids = self._get_centroids(self.clusters)

            if self._is_converged(centroids_old, self.centroids):
                break

            if self.plot_steps:
                self.plot()

        # Clasificación de los samples como indices en sus clusters 
        return self._get_cluster_labels(self.clusters)


    def _get_cluster_labels(self, clusters):
        # cada sample obtendrá una etiqueta del cluster al que le corresponde
        labels = np.empty(self.n_samples)
        for cluster_idx, cluster in enumerate(clusters):
            for sample_idx in cluster:
                labels[sample_idx] = cluster_idx

        return labels


    def _create_clusters(self, centroids):
        # Asignación de los samples a el centroide más cercano
        clusters = [[] for _ in range(self.K)]
        for idx, sample in enumerate(self.X):
            centroid_idx = self._closest_centroid(sample, centroids)
            clusters[centroid_idx].append(idx)
        return clusters

    def _closest_centroid(self, sample, centroids):
        # distancia del punto actual con cada centroide
        distances = [euclidean_distance(sample, point) for point in centroids]
        closest_idx = np.argmin(distances)
        return closest_idx


    def _get_centroids(self, clusters):
        # asignación de la media de los centroides de cada cluster
        centroids = np.zeros((self.K, self.n_features))
        for cluster_idx, cluster in enumerate(clusters):
            cluster_mean = np.mean(self.X[cluster], axis=0)
            centroids[cluster_idx] = cluster_mean
        return centroids

    def _is_converged(self, centroids_old, centroids):
        # distancias entre el viejo y nuevos centroides, para todos los centroide
        distances = [euclidean_distance(centroids_old[i], centroids[i]) for i in range(self.K)]
        return sum(distances) == 0

    def plot(self):
        fig, ax = plt.subplots(figsize=(12, 8))

        for i, index in enumerate(self.clusters):
            point = self.X[index].T
            ax.scatter(*point)

        for point in self.centroids:
            ax.scatter(*point, marker="x", color="black", linewidth=2)

        plt.show()


In [ ]:
np.random.seed(42)
from sklearn.datasets import make_blobs

X, y = make_blobs(
    centers=3, n_samples=500, n_features=2, shuffle=True, random_state=40
)
print(X.shape)

clusters = len(np.unique(y))
print(clusters)

k = KMeans(K=clusters, max_iters=150, plot_steps=True)
y_pred = k.predict(X)

k.plot()

## Criterios de Elección de Grupos

> Criterio del codo

> Criterio del gradiente

## Ejemplo 1

In [ ]:
#%% Generar datos aleatorios
X, Y = make_blobs(n_samples=1500, random_state=5)

In [ ]:
X

In [ ]:
Y

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(X[:,0],X[:,1])
plt.show()

In [ ]:
from sklearn.cluster import KMeans
#%% Aplicar el algoritmo Kmeans
model = KMeans(n_clusters=10, random_state=5, init='random')
model = model.fit(X)
grupos = model.predict(X)
centroides = model.cluster_centers_

In [ ]:
grupos

In [ ]:
centroides

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(X[:,0],X[:,1], c=grupos)
plt.plot(centroides[:,0], centroides[:,1], 'x')
plt.show()

In [ ]:
model.inertia_

In [ ]:
#%% Criterio de selección del codo
ngroup = 15
inercias = np.zeros(ngroup)

for k in np.arange(1,ngroup):
    model = KMeans(n_clusters = k, random_state=5, init='random')
    model = model.fit(X)
    inercias[k] = model.inertia_
    
plt.figure(figsize=(6,4))
plt.plot(np.arange(1,ngroup), inercias[1:])
plt.xlabel('Número de grupos')
plt.ylabel('Inercias')
    

In [ ]:
#%% Definiendo el número de grupos optimos
#%% Aplicar el algoritmo Kmeans con 2 grupos
model = KMeans(n_clusters=2, random_state=5, init='random')
model = model.fit(X)
grupos = model.predict(X)
centroides = model.cluster_centers_

plt.figure(figsize=(6,4))
plt.scatter(X[:,0],X[:,1], c=grupos)
plt.plot(centroides[:,0], centroides[:,1], 'x')
plt.show()

In [ ]:
# Función del criterio de selección de grupos (criterio del codo)
def criterio_codo_kmeans(ngroup, data):
    inercias = np.zeros(ngroup)

    for k in np.arange(1,ngroup):
        model = KMeans(n_clusters = k, random_state=5, init='random')
        model = model.fit(data)
        inercias[k] = model.inertia_
        
    plt.figure(figsize=(6,4))
    plt.plot(np.arange(1,ngroup), inercias[1:])
    plt.xlabel('Número de grupos')
    plt.ylabel('Inercias')
    
    

In [ ]:
criterio_codo_kmeans(5,X)

In [ ]:
# Función del criterio de selección de grupos (criterio del gradiente)
def criterio_gradiente_kmeans(ngroup, data):
    inercias = np.zeros(ngroup)

    for k in np.arange(1,ngroup):
        model = KMeans(n_clusters = k, random_state=5, init='random')
        model = model.fit(data)
        inercias[k] = model.inertia_
    
    #Derivar las inercias y graficar el gradiente
    gradiente =  np.diff(inercias)
    plt.figure(figsize=(6,4))
    plt.plot(np.arange(1,ngroup), gradiente)
    plt.xlabel('Número de grupos')
    plt.ylabel('Inercias')
    

In [ ]:
criterio_gradiente_kmeans(15,X)

### Discretización

In [ ]:
from sklearn.linear_model import LinearRegression

df=pd.read_csv('dataKmeans.csv')
plt.scatter(df.x,df.y,s=5)
plt.grid()

In [ ]:
df.hist(bins=50)

In [ ]:
df.x.hist(bins=50)

In [ ]:
criterio_codo_kmeans(ngroup=10, data=df[['x']])

In [ ]:
# Centroides
k=3
ctr=np.random.uniform(df.x.min(),df.x.max(),k)
ctr

In [ ]:
#Algoritmo Kmeans básico
dif=[]
for c_i in ctr:
    dif.append(np.abs(c_i-df[['x']].values))
distancias=np.concatenate(dif,axis=1)
grupos=np.argmin(distancias,axis=1)
df_copia=df.copy()
df_copia['kmeans']=grupos
df_copia.groupby('kmeans')['x'].mean()

In [ ]:
df_copia

In [ ]:
df_copia['kmeans'].value_counts()

In [ ]:
ctr_anterior=np.ones(k)*np.inf
eps=1e-6
while(np.abs(ctr-ctr_anterior).sum()>eps): # Minkowski con p=1, criterio de paro
    dif=[]
    for c_i in ctr:
        dif.append(np.abs(c_i-df[['x']].values))
    distancias=np.concatenate(dif,axis=1)
    grupos=np.argmin(distancias,axis=1)
    df_copia=df.copy()
    df_copia['kmeans']=grupos
    ctr_anterior=ctr.copy()
    ctr=df_copia.groupby('kmeans')['x'].mean().values
df['kmeans']=grupos
df.groupby('kmeans')['x'].hist(bins='auto')
ctr

#### Regresión lineal de los datos discretizados

In [ ]:
# Modelo con datos sin discretizar
lin_SD=LinearRegression()
lin_SD.fit(df[['x']],df['y'])
predict_SD=lin_SD.predict(df[['x']])

# Modelo con datos discretizados con K-means
lin_kmeans=LinearRegression()
x=ctr.reshape(-1,1)
y=df.groupby('kmeans')['y'].mean()
lin_kmeans.fit(x,y)
predict_kmeans=lin_kmeans.predict(df[['x']].values)

# Gráfica
plt.scatter(df.x,df.y,s=5,c='k')
plt.plot(df.x,predict_SD,'g',label='Regresión sin discretizar',lw=3)
plt.plot(df.x,predict_kmeans,'r',label='Regresión con K-means')
plt.legend()
plt.grid()

In [ ]:
# Modelo con datos sin discretizar
lin_SD.coef_,lin_SD.intercept_

In [ ]:
# Modelo con datos discretizados usando K-means
lin_kmeans.coef_,lin_kmeans.intercept_

In [ ]:
lin_SD.score(df[['x']],df['y']),lin_kmeans.score(df[['x']].values,df['y'])

In [ ]:
plt.scatter(df.x,df.y,c='k',s=5)
lin_centroides=[]
for c_i in range(k):
    x=df[df['kmeans']==c_i][['x']]
    y=df[df['kmeans']==c_i]['y']
    lin_centroides.append(LinearRegression())
    lin_centroides[-1].fit(x,y)

    plt.plot(x,lin_centroides[-1].predict(x),label='Grupo '+str(c_i))
plt.legend()
plt.grid()